In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install duckdb -q

import duckdb
from pathlib import Path

# Create a temp folder DuckDB can spill to disk if it runs low on RAM
Path('/content/duck_tmp').mkdir(exist_ok=True)

con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET temp_directory='/content/duck_tmp'")
con.execute("SET preserve_insertion_order=false")  # lower memory for big writes
print("DuckDB ready.")

DuckDB ready.


In [6]:
DATA_DIR = Path('/content/drive/Shareddrives/CapStone Project Carrefour/Data')

ticket_csv        = DATA_DIR / 'ie_linea_ticket.csv'
articulos_csv     = DATA_DIR / 'ie_maestra_articulos.csv'
ticket_parquet    = DATA_DIR / 'ie_linea_ticket.parquet'
articulos_parquet = DATA_DIR / 'ie_maestra_articulos.parquet'

for f in (ticket_csv, articulos_csv):
    print(f"{'OK ' if f.exists() else 'MISSING'} -> {f}")

OK  -> /content/drive/Shareddrives/CapStone Project Carrefour/Data/ie_linea_ticket.csv
OK  -> /content/drive/Shareddrives/CapStone Project Carrefour/Data/ie_maestra_articulos.csv


In [ ]:
# Runs once. Slow (reads 25 GB off Drive) but RAM-safe — it streams.
# Skip this cell on future sessions once the parquet exists.
if not ticket_parquet.exists():
    con.execute(f"""
        COPY (SELECT * FROM read_csv_auto('{ticket_csv}'))
        TO '{ticket_parquet}' (FORMAT PARQUET)
    """)
    print("ticket parquet written:", ticket_parquet)
else:
    print("ticket parquet already exists — skipping.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ticket parquet written: /content/drive/Shareddrives/CapStone Project Carrefour/Data/ie_linea_ticket.parquet


In [ ]:
if not articulos_parquet.exists():
    con.execute(f"""
        COPY (SELECT * FROM read_csv_auto('{articulos_csv}'))
        TO '{articulos_parquet}' (FORMAT PARQUET)
    """)
    print("articulos parquet written:", articulos_parquet)
else:
    print("articulos parquet already exists — skipping.")

articulos parquet written: /content/drive/Shareddrives/CapStone Project Carrefour/Data/ie_maestra_articulos.parquet


In [ ]:
T = f"'{ticket_parquet}'"  # shorthand for the queries below

print("=== ie_linea_ticket ===")
print("\nSchema:")
display(con.execute(f"DESCRIBE SELECT * FROM {T}").df())

print("\nRow count:")
display(con.execute(f"SELECT count(*) AS n_rows FROM {T}").df())

print("\nFirst 20 rows:")
display(con.execute(f"SELECT * FROM {T} LIMIT 20").df())

=== ie_linea_ticket ===

Schema:


,column_name,column_type,null,key,default,extra
0,idempres,VARCHAR,YES,None,None,None
1,fecha,DATE,YES,None,None,None
2,hora,VARCHAR,YES,None,None,None
3,ticket,VARCHAR,YES,None,None,None
4,cliente,VARCHAR,YES,None,None,None
5,idarticu,VARCHAR,YES,None,None,None
6,unidades,BIGINT,YES,None,None,None
7,importe,DOUBLE,YES,None,None,None
8,idpromoc,VARCHAR,YES,None,None,None
9,idtiprod,BIGINT,YES,None,None,None



Row count:


,n_rows
0,191017715



First 20 rows:


,idempres,fecha,hora,ticket,cliente,idarticu,unidades,importe,idpromoc,idtiprod
0,07,2022-02-27,2152,2022-02-2700490049008008436,fad0e76cd828a6af762c45906856d5f4a898c5d43b117c...,242631,3,7.38,Promo,2
1,06,2022-04-20,0933,2022-04-2028742874132006808,063e1081001bbf51f2364fe6f8642adc34daa35b135846...,000148,1,0.63,No promo,2
2,07,2022-03-07,1111,2022-03-0702030203023004621,ab50bc17729cb86c32c8cb430af82134c4b53c62e56c5d...,297595,1,3.29,No promo,2
3,07,2022-04-03,1205,2022-04-0302030203021003381,20321c10ac73c769833cf0253f949b75eeac0171457267...,497517,1,3.00,Promo,2
4,07,2022-06-03,1309,2022-06-0300550055010004535,542311cf65909c96d42e084e252cbe2242df5dd7016758...,584494,1,4.13,No promo,1
5,07,2022-02-18,1237,2022-02-1800960096032005984,46e78fbd131aff9a952ba78d3e9dec9e8043b03ba2ffdc...,632052,2,0.20,No promo,1
6,07,2022-05-06,1231,2022-05-0600520052019008363,fee3cccba6704f1930a41dcafb9015582909ee71d18ccd...,487580,1,7.09,No promo,2
7,02,2022-04-12,1829,2022-04-1251456318136000428,a006a5743b317068b147c72f3ef4e9d8165e61b0e5cdd5...,235347,1,0.99,No promo,1
8,07,2022-01-05,1508,2022-01-0500700070115002012,ba6acedb9b1aace9d46c6abadaa53c3eecde007dbda6db...,128222,1,3.65,No promo,1
9,06,2022-02-10,1449,2022-02-1016381638131007537,3401897ee667a07a8e473b2c77851fd4f01197fb7d56ac...,035923,1,2.89,No promo,2


In [ ]:
# Get column names from the schema, then build per-column checks in SQL
cols = con.execute(f"DESCRIBE SELECT * FROM {T}").df()['column_name'].tolist()

null_expr   = ", ".join([f"sum(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) AS \"{c}\"" for c in cols])
unique_expr = ", ".join([f"count(DISTINCT \"{c}\") AS \"{c}\"" for c in cols])

print("Missing values per column:")
display(con.execute(f"SELECT {null_expr} FROM {T}").df().T.rename(columns={0: 'n_missing'}))

print("\nUnique values per column:")
display(con.execute(f"SELECT {unique_expr} FROM {T}").df().T.rename(columns={0: 'n_unique'}))

Missing values per column:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_missing
idempres,0.0
fecha,0.0
hora,0.0
ticket,0.0
cliente,0.0
idarticu,0.0
unidades,0.0
importe,0.0
idpromoc,0.0
idtiprod,0.0



Unique values per column:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_unique
idempres,4
fecha,181
hora,1440
ticket,20026724
cliente,1482715
idarticu,117701
unidades,858
importe,19440
idpromoc,2
idtiprod,3


In [ ]:
import pandas as pd

df_articulos = pd.read_parquet(articulos_parquet)

print(f"ie_maestra_articulos — {df_articulos.shape[0]:,} rows × {df_articulos.shape[1]} cols")
display(df_articulos.head())
df_articulos.info()

print("\nMissing values per column:")
display(df_articulos.isna().sum().to_frame('n_missing').assign(
    pct=lambda x: (x.n_missing / len(df_articulos) * 100).round(2)))

print("\nDuplicate rows:", df_articulos.duplicated().sum())
print("\nUnique values per column:")
display(df_articulos.nunique().sort_values().to_frame('n_unique'))

ie_maestra_articulos — 893,944 rows × 4 cols


,idarticu,desc_larga_articulo,idsector,desc_sector
0,966557,GUISO DE VERDURAS LITORAL 415 GR,0001,P.G.C.
1,909163,LA HISTORIA DE LOS 3 CERDITOS ANAYA,0003,BAZAR
2,594029,MUU (SUSAETA),0003,BAZAR
3,086412,ALAS DE MOSCA PARA ÁNGEL ANAYA EDUCACIÓN,0003,BAZAR
4,507503,CAMISETA NINO MANGA LARGA SPIDERMAN TINTA META...,0006,TEXTIL


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893944 entries, 0 to 893943
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   idarticu             893944 non-null  object
 1   desc_larga_articulo  893944 non-null  object
 2   idsector             893944 non-null  object
 3   desc_sector          893944 non-null  object
dtypes: object(4)
memory usage: 27.3+ MB

Missing values per column:


,n_missing,pct
idarticu,0,0.0
desc_larga_articulo,0,0.0
idsector,0,0.0
desc_sector,0,0.0



Duplicate rows: 0

Unique values per column:


,n_unique
desc_sector,6
idsector,6
desc_larga_articulo,823031
idarticu,893944


In [ ]:
# describe() works on the parquet via DuckDB without loading it
print("Numeric summary — ie_linea_ticket:")
display(con.execute(f"SUMMARIZE SELECT * FROM {T}").df())

print("\nNumeric summary — ie_maestra_articulos:")
display(df_articulos.describe().T)

Numeric summary — ie_linea_ticket:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,idempres,VARCHAR,02,11,4,None,None,None,None,None,191017715,0.0
1,fecha,DATE,2022-01-01,2022-06-30,179,2022-04-01 01:31:11.462824,None,2022-02-15,2022-03-31,2022-05-17,191017715,0.0
2,hora,VARCHAR,0000,2359,1649,None,None,None,None,None,191017715,0.0
3,ticket,VARCHAR,2022-01-0140133647013005558,2022-06-3083203114132004467,22983945,None,None,None,None,None,191017715,0.0
4,cliente,VARCHAR,00001548c54a65f239f9562e70952b2589fb43e899d472...,ffffe0601a99cb314df5d9676d4327a49310f02c79b9aa...,1604643,None,None,None,None,None,191017715,0.0
5,idarticu,VARCHAR,000001,992451,117551,None,None,None,None,None,191017715,0.0
6,unidades,BIGINT,1,78624,1022,1.576246904639185,32.12787474762679,1,1,1,191017715,0.0
7,importe,DOUBLE,-60.0,48384.0,20320,3.6485648080959012,26.46784798871832,1.250740182934361,2.064202428996128,3.734585160153071,191017715,0.0
8,idpromoc,VARCHAR,No promo,Promo,2,None,None,None,None,None,191017715,0.0
9,idtiprod,BIGINT,0,2,3,1.4870315457390955,0.5292898071963975,1,2,2,191017715,0.0



Numeric summary — ie_maestra_articulos:


,count,unique,top,freq
idarticu,893944,893944,322715,1
desc_larga_articulo,893944,823031,ACCESORIOS DE MODA,24859
idsector,893944,6,0003,425050
desc_sector,893944,6,BAZAR,425050


# Data Prep — Cleaning, Basket Corpus, Vocabulary

Everything below is the user's deliverable for **Phase 0** of the Carrefour
pipeline: a validated basket-level corpus + product vocabulary that the next
teammate will feed into Item2Vec.

All heavy work runs in **DuckDB SQL** (streaming, out-of-core). Pandas only
materializes small summary results. Every filter logs row/basket counts
**before and after** so nothing is dropped silently.


In [7]:
# === CONFIG — every threshold lives here so the next person can audit/tune ===
import duckdb, pandas as pd, numpy as np, sys
from pathlib import Path
from datetime import datetime
import shutil, time, os

print("Library versions:")
print("  duckdb :", duckdb.__version__)
print("  pandas :", pd.__version__)
print("  numpy  :", np.__version__)
print("  python :", sys.version.split()[0])

# --- Paths ----------------------------------------------------------------
DATA_DIR        = Path("/content/drive/Shareddrives/CapStone Project Carrefour/Data")
TICKET_PARQUET  = DATA_DIR / "ie_linea_ticket.parquet"          # source (Drive)
ARTIC_PARQUET   = DATA_DIR / "ie_maestra_articulos.parquet"     # source (Drive)

# Local working copy of the ticket parquet — Drive FUSE is slow for repeated scans.
LOCAL_TICKET    = Path("/content/ie_linea_ticket.parquet")

# Final outputs (persist on Drive)
CLEAN_LINES_PQ  = DATA_DIR / "lineitems_clean.parquet"
CLEAN_BUCKET_DIR = DATA_DIR / "lineitems_clean_by_basket_part"
BASKET_DIR      = DATA_DIR / "basket_corpus_parts"
BASKET_GLOB     = BASKET_DIR / "*.parquet"
VOCAB_PQ        = DATA_DIR / "product_vocab.parquet"
QA_REPORT_MD    = DATA_DIR / "DATA_QUALITY_REPORT.md"
HANDOFF_MD      = DATA_DIR / "HANDOFF_to_embedding_step.md"

# --- Cleaning rules -------------------------------------------------------
RETURNS_RULE_SQL = "unidades > 0 AND importe > 0"   # drops refunds + zero lines
MIN_BASKET_SIZE  = 2                                # need ≥2 distinct products for co-occurrence
MAX_BASKET_SIZE  = None                             # no cap — just log distribution
RANDOM_SEED      = 42

# --- DuckDB settings ------------------------------------------------------
BASKET_PARTITIONS = 64                              # raise to 128 if Colab still OOMs
DUCK_MEM_LIMIT   = "8GB"                            # leave headroom below Colab RAM
DUCK_THREADS     = 2                                # lower per-query memory pressure
DUCK_TMP         = Path("/content/duck_tmp")
DUCK_TMP.mkdir(exist_ok=True)

con = duckdb.connect() # Initialize duckdb connection

con.execute(f"SET memory_limit='{DUCK_MEM_LIMIT}'")
con.execute(f"SET threads={DUCK_THREADS}")
con.execute(f"SET temp_directory='{DUCK_TMP}'")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SELECT setseed({RANDOM_SEED/100})")

print("\nConfig loaded. Outputs will be written to:", DATA_DIR)

Library versions:
  duckdb : 1.3.2
  pandas : 2.2.2
  numpy  : 2.0.2
  python : 3.12.13

Config loaded. Outputs will be written to: /content/drive/Shareddrives/CapStone Project Carrefour/Data


## Stage 1 — Stage the ticket parquet on local disk

Drive FUSE I/O is slow and we'll scan the ticket parquet several times (cleaning,
join check, basket build). Copy it once to `/content/` (local SSD), then point
DuckDB at the local copy. Skip if already there.


In [8]:
if LOCAL_TICKET.exists() and LOCAL_TICKET.stat().st_size == TICKET_PARQUET.stat().st_size:
    print(f"Local ticket parquet already staged → {LOCAL_TICKET}")
else:
    t0 = time.time()
    size_gb = TICKET_PARQUET.stat().st_size / 1e9
    print(f"Copying {size_gb:.1f} GB from Drive → /content ...")
    shutil.copy2(TICKET_PARQUET, LOCAL_TICKET)
    print(f"  done in {time.time()-t0:,.0f}s")

# All downstream SQL references this constant
TKT = f"'{LOCAL_TICKET}'"
ART = f"'{ARTIC_PARQUET}'"

print(con.execute(f"SELECT count(*) AS n_rows FROM {TKT}").df())


Copying 17.2 GB from Drive → /content ...
  done in 276s
      n_rows
0  191017715


## Stage 2 — Clean the line-item data (streaming, single pass)

Drops, in order, with row counts logged at each step:

1. rows with null `idarticu` or null `ticket`
2. returns / zero-value lines (`unidades > 0 AND importe > 0`)
3. exact-duplicate line rows

We do **not** filter rare products — that's the embedding person's `min_count`
call. Output: `lineitems_clean.parquet` (zstd).


In [9]:
# Single SQL pass with per-step counts via filtered SUMs (avoids multiple table scans).
DROP_LOG = {}

print("Counting per-step drops in one streaming scan ...")
t0 = time.time()
counts = con.execute(f"""
    SELECT
      count(*)                                                          AS n_raw,
      sum(CASE WHEN idarticu IS NULL OR ticket IS NULL THEN 1 ELSE 0 END) AS n_null_keys,
      sum(CASE WHEN (idarticu IS NOT NULL AND ticket IS NOT NULL)
                AND NOT (unidades > 0 AND importe > 0) THEN 1 ELSE 0 END)  AS n_return_or_zero
    FROM {TKT}
""").df().iloc[0]
print(f"  scan done in {time.time()-t0:,.0f}s")

DROP_LOG["raw_rows"]           = int(counts.n_raw)
DROP_LOG["drop_null_keys"]     = int(counts.n_null_keys)
DROP_LOG["drop_returns_zeros"] = int(counts.n_return_or_zero)

# Write the cleaned line items. DISTINCT removes exact duplicate rows.
if CLEAN_LINES_PQ.exists():
    print(f"\n{CLEAN_LINES_PQ.name} already exists — skipping write.")
else:
    print("\nWriting cleaned line items (DISTINCT, zstd) ...")
    t0 = time.time()
    con.execute(f"""
        COPY (
            SELECT DISTINCT
                ticket    AS basket_id,
                cliente   AS customer_id,
                fecha     AS date,
                idempres  AS store_id,
                idarticu  AS product_id,
                unidades  AS quantity,
                importe   AS amount,
                idpromoc  AS promo_flag,
                idtiprod  AS product_type
            FROM {TKT}
            WHERE idarticu IS NOT NULL
              AND ticket   IS NOT NULL
              AND {RETURNS_RULE_SQL}
        )
        TO '{CLEAN_LINES_PQ}' (FORMAT PARQUET, COMPRESSION 'zstd')
    """)
    print(f"  done in {time.time()-t0:,.0f}s → {CLEAN_LINES_PQ}")

CLEAN = f"'{CLEAN_LINES_PQ}'"
n_clean = con.execute(f"SELECT count(*) FROM {CLEAN}").fetchone()[0]
DROP_LOG["clean_rows"]    = n_clean
DROP_LOG["drop_exact_dup"] = (DROP_LOG["raw_rows"]
                              - DROP_LOG["drop_null_keys"]
                              - DROP_LOG["drop_returns_zeros"]
                              - n_clean)

print("\n── Line-item drop log ──")
for k, v in DROP_LOG.items():
    pct = v / DROP_LOG["raw_rows"] * 100
    print(f"  {k:<22} {v:>14,}   ({pct:5.2f}%)")

assert n_clean > 0, "Cleaned table is empty — check filter rules."
assert DROP_LOG["drop_exact_dup"] >= 0, "Negative dedupe count → arithmetic mistake."


Counting per-step drops in one streaming scan ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  scan done in 64s

Writing cleaned line items (DISTINCT, zstd) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  done in 2,660s → /content/drive/Shareddrives/CapStone Project Carrefour/Data/lineitems_clean.parquet

── Line-item drop log ──
  raw_rows                  191,017,715   (100.00%)
  drop_null_keys                      0   ( 0.00%)
  drop_returns_zeros            655,196   ( 0.34%)
  clean_rows                190,362,519   (99.66%)
  drop_exact_dup                      0   ( 0.00%)


## Stage 3 — Validate the join to the article master

We don't drop unknown products (the embedding step can still learn from them
via co-occurrence), but we must **measure** how many we have and report it.


In [10]:
JOIN_COVERAGE = {}

print("Computing master-join coverage ...")
row = con.execute(f"""
    WITH lines  AS (SELECT product_id, basket_id FROM {CLEAN}),
         master AS (SELECT DISTINCT idarticu FROM {ART})
    SELECT
      count(*)                                                       AS n_lines,
      sum(CASE WHEN m.idarticu IS NULL THEN 1 ELSE 0 END)             AS n_lines_unknown,
      count(DISTINCT l.product_id)                                    AS n_distinct_products,
      count(DISTINCT CASE WHEN m.idarticu IS NULL THEN l.product_id END)
                                                                      AS n_distinct_unknown,
      count(DISTINCT l.basket_id)                                     AS n_baskets,
      count(DISTINCT CASE WHEN m.idarticu IS NULL THEN l.basket_id END)
                                                                      AS n_baskets_touching_unknown
    FROM lines l LEFT JOIN master m ON l.product_id = m.idarticu
""").df().iloc[0]

JOIN_COVERAGE.update({k: int(v) for k, v in row.items()})
JOIN_COVERAGE["pct_lines_unknown"]    = row.n_lines_unknown / row.n_lines * 100
JOIN_COVERAGE["pct_products_unknown"] = row.n_distinct_unknown / row.n_distinct_products * 100
JOIN_COVERAGE["pct_baskets_touching_unknown"] = row.n_baskets_touching_unknown / row.n_baskets * 100

print("\n── Master-join coverage ──")
for k, v in JOIN_COVERAGE.items():
    print(f"  {k:<32} {v:>14,.4f}" if isinstance(v, float) else f"  {k:<32} {v:>14,}")


Computing master-join coverage ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


── Master-join coverage ──
  n_lines                             190,362,519
  n_lines_unknown                               0
  n_distinct_products                     117,621
  n_distinct_unknown                            0
  n_baskets                            20,026,612
  n_baskets_touching_unknown                    0
  pct_lines_unknown                        0.0000
  pct_products_unknown                     0.0000
  pct_baskets_touching_unknown             0.0000


## Stage 4 — Build the basket corpus (the Item2Vec sentences)

Group cleaned line items by `basket_id` into a **list of distinct product IDs**.
The previous single `GROUP BY basket_id` can exceed Colab RAM on ~190M lines,
so this stage first buckets cleaned rows by `hash(basket_id)` and then aggregates
one bucket at a time. The final corpus is a parquet directory, not one giant file.

We carry `customer_id` and `date` so the next phase (customer aggregation) has
them. Drop baskets with fewer than `MIN_BASKET_SIZE=2` distinct products.

No upper cap — `MAX_BASKET_SIZE = None`. We log the size distribution so the
embedding teammate can decide if/where to cap.


In [ ]:
BASKET_LOG = {}

print("Building basket corpus in hash partitions ...")
t0 = time.time()

# Step 4a: bucket the cleaned line rows once. This avoids scanning the full
# cleaned file once per basket partition and guarantees every basket lands in
# exactly one later aggregation job.
bucket_files = list(CLEAN_BUCKET_DIR.glob("basket_part=*/*.parquet")) if CLEAN_BUCKET_DIR.exists() else []
bucket_parts = sorted({int(p.parent.name.split("=", 1)[1]) for p in bucket_files})
if len(bucket_parts) == BASKET_PARTITIONS:
    print(f"  clean line buckets already exist -> {CLEAN_BUCKET_DIR}")
elif bucket_files:
    raise RuntimeError(
        f"Found only {len(bucket_parts)} of {BASKET_PARTITIONS} clean buckets in {CLEAN_BUCKET_DIR}. "
        "Delete that incomplete directory in Drive, then rerun this cell."
    )
else:
    print(f"  writing {BASKET_PARTITIONS} clean line buckets -> {CLEAN_BUCKET_DIR}")
    con.execute(f"""
        COPY (
            SELECT
                *,
                hash(basket_id) % {BASKET_PARTITIONS} AS basket_part
            FROM {CLEAN}
        )
        TO '{CLEAN_BUCKET_DIR}'
        (FORMAT PARQUET, COMPRESSION 'zstd', PARTITION_BY (basket_part))
    """)
    print(f"  clean buckets written in {time.time()-t0:,.0f}s")

# Step 4b: aggregate one bucket at a time. Each query now handles roughly
# 1 / BASKET_PARTITIONS of the baskets instead of all ~20M groups at once.
BASKET_DIR.mkdir(exist_ok=True)
for part in range(BASKET_PARTITIONS):
    part_path = BASKET_DIR / f"part_{part:03d}.parquet"
    if part_path.exists():
        print(f"  basket part {part:03d}/{BASKET_PARTITIONS-1:03d} exists — skipping")
        continue

    source_glob = CLEAN_BUCKET_DIR / f"basket_part={part}" / "*.parquet"
    print(f"  aggregating basket part {part:03d}/{BASKET_PARTITIONS-1:03d} ...")
    part_t0 = time.time()
    con.execute(f"""
        COPY (
            WITH line_products AS (
                SELECT
                    basket_id,
                    any_value(customer_id) AS customer_id,
                    any_value(date)        AS date,
                    any_value(store_id)    AS store_id,
                    product_id
                FROM read_parquet('{source_glob}')
                GROUP BY basket_id, product_id
            ),
            agg AS (
                SELECT
                    basket_id,
                    any_value(customer_id) AS customer_id,
                    any_value(date)        AS date,
                    any_value(store_id)    AS store_id,
                    list(product_id ORDER BY product_id) AS products
                FROM line_products
                GROUP BY basket_id
            )
            SELECT
                basket_id,
                customer_id,
                date,
                store_id,
                products,
                len(products) AS basket_size
            FROM agg
            WHERE len(products) >= {MIN_BASKET_SIZE}
        )
        TO '{part_path}' (FORMAT PARQUET, COMPRESSION 'zstd')
    """)
    print(f"    done in {time.time()-part_t0:,.0f}s")

BSK = f"read_parquet('{BASKET_GLOB}')"

# Count baskets pre-filter (all groups) vs post-filter (what we wrote) so the
# drop log includes the size-floor drop.
n_total_baskets = (
    JOIN_COVERAGE.get("n_baskets")
    if "JOIN_COVERAGE" in globals() and "n_baskets" in JOIN_COVERAGE
    else con.execute(f"SELECT count(DISTINCT basket_id) FROM {CLEAN}").fetchone()[0]
)
n_kept_baskets  = con.execute(f"SELECT count(*) FROM {BSK}").fetchone()[0]

BASKET_LOG["baskets_total"]       = n_total_baskets
BASKET_LOG["baskets_kept"]        = n_kept_baskets
BASKET_LOG["baskets_dropped_lt2"] = n_total_baskets - n_kept_baskets

print(f"\nBasket parquet dataset written in {time.time()-t0:,.0f}s -> {BASKET_DIR}")
print("\n── Basket drop log ──")
for k, v in BASKET_LOG.items():
    print(f"  {k:<25} {v:>14,}")

# Basket-size distribution
print("\nBasket-size distribution:")
SIZE_DIST = con.execute(f"""
    SELECT
        min(basket_size) AS min,
        approx_quantile(basket_size, 0.25) AS p25,
        approx_quantile(basket_size, 0.50) AS p50,
        approx_quantile(basket_size, 0.75) AS p75,
        approx_quantile(basket_size, 0.95) AS p95,
        approx_quantile(basket_size, 0.99) AS p99,
        approx_quantile(basket_size, 0.999) AS p999,
        max(basket_size) AS max,
        avg(basket_size) AS mean
    FROM {BSK}
""").df()
display(SIZE_DIST)


## Stage 5 — Build the product vocabulary

- distinct `product_id` → contiguous integer index `0..V-1`
- join master for `desc_larga_articulo` (name) and `desc_sector` (category)
- per-product `basket_frequency` = number of baskets containing the product

The frequency column lets the embedding teammate pick `min_count` themselves.


In [ ]:
print("Building product vocabulary ...")
t0 = time.time()

if VOCAB_PQ.exists():
    print(f"  {VOCAB_PQ.name} already exists — skipping write.")
else:
    con.execute(f"""
        COPY (
            WITH freq AS (
                SELECT product_id, count(DISTINCT basket_id) AS basket_frequency
                FROM (
                    SELECT unnest(products) AS product_id, basket_id FROM {BSK}
                )
                GROUP BY product_id
            ),
            indexed AS (
                SELECT
                    product_id,
                    basket_frequency,
                    (row_number() OVER (ORDER BY basket_frequency DESC, product_id) - 1) AS product_index
                FROM freq
            )
            SELECT
                i.product_index,
                i.product_id,
                m.desc_larga_articulo AS product_name,
                m.idsector            AS sector_id,
                m.desc_sector         AS sector_name,
                i.basket_frequency
            FROM indexed i
            LEFT JOIN read_parquet({ART}) m ON i.product_id = m.idarticu
            ORDER BY i.product_index
        )
        TO '{VOCAB_PQ}' (FORMAT PARQUET, COMPRESSION 'zstd')
    """)
    print(f"  done in {time.time()-t0:,.0f}s → {VOCAB_PQ}")

VOC = f"'{VOCAB_PQ}'"
V = con.execute(f"SELECT count(*) FROM {VOC}").fetchone()[0]
print(f"\nVocabulary size V = {V:,}")

print("\nTop 10 most frequent products:")
display(con.execute(f"SELECT * FROM {VOC} ORDER BY basket_frequency DESC LIMIT 10").df())

print("\nFrequency tail — singletons & rare:")
display(con.execute(f"""
    SELECT
        sum(CASE WHEN basket_frequency = 1 THEN 1 ELSE 0 END)  AS appears_in_1_basket,
        sum(CASE WHEN basket_frequency <= 5 THEN 1 ELSE 0 END) AS appears_in_le_5,
        sum(CASE WHEN basket_frequency <= 10 THEN 1 ELSE 0 END) AS appears_in_le_10,
        sum(CASE WHEN basket_frequency <= 50 THEN 1 ELSE 0 END) AS appears_in_le_50,
        count(*) AS total_products
    FROM {VOC}
""").df())


## Stage 6 — Data Quality Report

Writes a markdown report covering every drop count, basket-size distribution,
vocabulary stats, and master-join coverage. This is the evidence that the
corpus is sound.


In [ ]:
# Pull a few small summaries DuckDB doesn't already have in memory
size_row = SIZE_DIST.iloc[0].to_dict()
date_row = con.execute(f"SELECT min(date) AS min_date, max(date) AS max_date FROM {BSK}").df().iloc[0]
n_customers = con.execute(f"SELECT count(DISTINCT customer_id) FROM {BSK}").fetchone()[0]
freq_tail = con.execute(f"""
    SELECT
        sum(CASE WHEN basket_frequency = 1 THEN 1 ELSE 0 END)  AS appears_in_1_basket,
        sum(CASE WHEN basket_frequency <= 5 THEN 1 ELSE 0 END) AS appears_in_le_5,
        sum(CASE WHEN basket_frequency <= 10 THEN 1 ELSE 0 END) AS appears_in_le_10,
        sum(CASE WHEN basket_frequency <= 50 THEN 1 ELSE 0 END) AS appears_in_le_50,
        count(*) AS total_products
    FROM {VOC}
""").df().iloc[0]

report = f"""# Data Quality Report — Carrefour Basket Corpus

Generated: {datetime.utcnow().isoformat()}Z
Source: `{TICKET_PARQUET.name}` ({DROP_LOG['raw_rows']:,} raw rows)
Basket corpus parquet dataset: `{BASKET_DIR.name}/` ({BASKET_PARTITIONS} parts)

## Line-item cleaning lineage

| Stage | Rows | % of raw |
|---|---:|---:|
| Raw lines | {DROP_LOG['raw_rows']:,} | 100.00% |
| Dropped — null product / null basket key | {DROP_LOG['drop_null_keys']:,} | {DROP_LOG['drop_null_keys']/DROP_LOG['raw_rows']*100:.2f}% |
| Dropped — returns / zero-value (`{RETURNS_RULE_SQL}`) | {DROP_LOG['drop_returns_zeros']:,} | {DROP_LOG['drop_returns_zeros']/DROP_LOG['raw_rows']*100:.2f}% |
| Dropped — exact duplicate rows | {DROP_LOG['drop_exact_dup']:,} | {DROP_LOG['drop_exact_dup']/DROP_LOG['raw_rows']*100:.2f}% |
| **Clean line items** | **{DROP_LOG['clean_rows']:,}** | **{DROP_LOG['clean_rows']/DROP_LOG['raw_rows']*100:.2f}%** |

## Basket construction lineage

| Stage | Baskets |
|---|---:|
| Distinct baskets in cleaned lines | {BASKET_LOG['baskets_total']:,} |
| Dropped — `<{MIN_BASKET_SIZE}` distinct products | {BASKET_LOG['baskets_dropped_lt2']:,} |
| **Basket corpus (final)** | **{BASKET_LOG['baskets_kept']:,}** |

Date range: **{date_row.min_date} → {date_row.max_date}**
Distinct customers in corpus: **{n_customers:,}**
Basket-size cap applied: **{MAX_BASKET_SIZE}** (None = no cap)

## Basket-size distribution

| min | p25 | p50 | p75 | p95 | p99 | p99.9 | max | mean |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| {int(size_row['min'])} | {int(size_row['p25'])} | {int(size_row['p50'])} | {int(size_row['p75'])} | {int(size_row['p95'])} | {int(size_row['p99'])} | {int(size_row['p999'])} | {int(size_row['max'])} | {size_row['mean']:.2f} |

## Vocabulary

- Vocabulary size **V = {V:,}** distinct products in final corpus
- Products in master file: {893944:,} (superset — many unsold)
- Products appearing in exactly **1 basket**: {int(freq_tail.appears_in_1_basket):,} ({freq_tail.appears_in_1_basket/V*100:.1f}%)
- Products appearing in **≤5 baskets**: {int(freq_tail.appears_in_le_5):,} ({freq_tail.appears_in_le_5/V*100:.1f}%)
- Products appearing in **≤10 baskets**: {int(freq_tail.appears_in_le_10):,} ({freq_tail.appears_in_le_10/V*100:.1f}%)
- Products appearing in **≤50 baskets**: {int(freq_tail.appears_in_le_50):,} ({freq_tail.appears_in_le_50/V*100:.1f}%)

Use this tail to choose `min_count` in Item2Vec.

## Master-join coverage

| Metric | Value |
|---|---:|
| Cleaned line rows | {JOIN_COVERAGE['n_lines']:,} |
| Lines whose product_id is **not** in master | {JOIN_COVERAGE['n_lines_unknown']:,} ({JOIN_COVERAGE['pct_lines_unknown']:.4f}%) |
| Distinct products in cleaned data | {JOIN_COVERAGE['n_distinct_products']:,} |
| Distinct products **not** in master | {JOIN_COVERAGE['n_distinct_unknown']:,} ({JOIN_COVERAGE['pct_products_unknown']:.4f}%) |
| Baskets touching ≥1 unknown product | {JOIN_COVERAGE['n_baskets_touching_unknown']:,} ({JOIN_COVERAGE['pct_baskets_touching_unknown']:.4f}%) |

Unknown products keep their `product_id` in the vocabulary; their
`product_name` / `sector_name` are NULL. The embedding teammate may still
learn vectors for them — they just don't have human-readable labels.

## Anomalies & decisions

- `importe` has negative values in raw data → confirmed returns/refunds; removed.
- `unidades` has extreme tail (max≈78k in raw); the size-floor and downstream
  min_count tuning will handle outlier baskets without aggressive truncation.
- Master has no brand / private-label / organic columns → not provided to Phase 1.
- `idtiprod` (3 values) and `idpromoc` (2 values) preserved in cleaned lines
  for potential future use; not in the basket corpus.

## Config used

```
RETURNS_RULE_SQL = "{RETURNS_RULE_SQL}"
MIN_BASKET_SIZE  = {MIN_BASKET_SIZE}
MAX_BASKET_SIZE  = {MAX_BASKET_SIZE}
RANDOM_SEED      = {RANDOM_SEED}
BASKET_PARTITIONS = {BASKET_PARTITIONS}
DUCK_MEM_LIMIT   = "{DUCK_MEM_LIMIT}"
DUCK_THREADS     = {DUCK_THREADS}
```
"""
QA_REPORT_MD.write_text(report)
print(f"Report written → {QA_REPORT_MD}")
print(f"  ({len(report):,} chars)")


## Stage 7 — Handoff document for the embedding teammate


In [ ]:
handoff = f"""# Handoff — to the Item2Vec embedding step

This document is for the teammate who owns **Phase 1 (product embedding)**.
The data-prep phase is done. You inherit two parquet datasets, one vocabulary
file, and a QA report.

## Files (all in `Data/`)

| File | Purpose |
|---|---|
| `basket_corpus_parts/` | **YOUR INPUT.** Partitioned parquet dataset. One row per basket; column `products` is the list of distinct product IDs = the Item2Vec "sentence". |
| `product_vocab.parquet` | product_id ↔ contiguous integer index ↔ name/sector ↔ basket_frequency |
| `lineitems_clean.parquet` | (optional) long-format cleaned lines, in case you need re-aggregation |
| `lineitems_clean_by_basket_part/` | intermediate bucketed line items used to build the basket corpus under Colab memory limits |
| `DATA_QUALITY_REPORT.md` | drop counts, distributions, coverage stats |

## Loading the corpus

```python
from pathlib import Path
import pyarrow.dataset as ds

DATA_DIR = Path("/content/drive/Shareddrives/CapStone Project Carrefour/Data")
CORPUS_DIR = DATA_DIR / "basket_corpus_parts"

class BasketSentences:
    def __init__(self, corpus_dir, batch_size=100_000):
        self.corpus_dir = corpus_dir
        self.batch_size = batch_size

    def __iter__(self):
        dataset = ds.dataset(self.corpus_dir, format="parquet")
        for batch in dataset.to_batches(columns=["products"], batch_size=self.batch_size):
            for products in batch.column("products").to_pylist():
                yield [str(product_id) for product_id in products]

sentences = BasketSentences(CORPUS_DIR)
```

`sentences` is a restartable streaming iterable for `gensim.models.Word2Vec`.
Do **not** call `pd.read_parquet(...).products.tolist()` on the full corpus;
that materializes all ~20M baskets in RAM.

## Mapping product_id ↔ index ↔ name

```python
import pandas as pd

vocab = pd.read_parquet("…/Data/product_vocab.parquet")
# columns: product_index, product_id, product_name, sector_id, sector_name, basket_frequency
id2name  = dict(zip(vocab.product_id, vocab.product_name))
id2index = dict(zip(vocab.product_id, vocab.product_index))
```

`product_index` is sorted by descending `basket_frequency` (index 0 = most frequent).
This is convenient for some embedding libraries but optional for gensim, which
builds its own internal vocab.

## Recommended starting point

- **Library:** `gensim.models.Word2Vec` (Item2Vec is Word2Vec on basket "sentences").
- **Window:** set large enough to cover the **entire basket** (e.g.
  `window = 9999`) — basket order is meaningless, every pair should co-occur.
- **sg = 1** (skip-gram) is the standard Item2Vec choice.
- **vector_size:** start 64 or 128.
- **negative:** 5–10 negative samples; `ns_exponent = 0.75` is the default.
- **epochs:** 5–10 to start; tune via downstream cluster quality.
- **min_count:** **YOUR CALL.** See the frequency tail in the QA report — many
  products appear in ≤5 baskets and likely cannot learn a good vector. A floor
  of 5–20 is typical. The vocabulary file's `basket_frequency` column is the
  source of truth.
- **workers:** equal to `os.cpu_count()`.

```python
import os
from gensim.models import Word2Vec
model = Word2Vec(
    sentences   = sentences,
    vector_size = 128,
    window      = 9999,
    sg          = 1,
    negative    = 10,
    min_count   = 10,        # tune per the frequency distribution
    epochs      = 5,
    workers     = os.cpu_count(),
    seed        = 42,
)
model.save("item2vec.model")
```

## Compute

- **gensim Item2Vec runs fine on CPU.** No GPU needed for Phase 1.
- **Later phases (UMAP, HDBSCAN)** are where GPU helps — request a GPU runtime
  for Phase 3+.

## Scope boundary

What I delivered:
- Clean line-item table with documented filter rules and drop lineage
- Basket corpus (the Item2Vec sentences) with customer_id + date preserved
- Product vocabulary with `basket_frequency`
- QA report with every drop count + master-join coverage

What I did **not** do (yours):
- Train embeddings / pick `min_count` / `vector_size` / `window`
- Customer-level frequency-weighted, time-decayed vector aggregation (Phase 2)
- Dimensionality reduction (Phase 3)
- HDBSCAN clustering (Phase 4)

## Caveats — please read

1. **Unknown products.** A small number of `product_id`s in the ticket data
   don't exist in the article master (their `product_name` is NULL in the
   vocab). Exact percentage is in the QA report. They are still in the corpus —
   you can keep them, drop them by joining the vocab, or include them but
   ignore them in your name-based qualitative checks.
2. **Single-item baskets dropped.** Baskets with `<{MIN_BASKET_SIZE}` distinct
   products were dropped (no co-occurrence signal). Count in the QA report.
3. **No basket cap.** I did not cap maximum basket size; the right-tail of the
   distribution is in the QA report. If you see wholesale-looking baskets
   degrading neighbor quality, consider filtering at your end.
4. **Returns removed.** Lines with `unidades ≤ 0` or `importe ≤ 0` were dropped
   as returns/refunds. If your modeling needs to know about returns, use
   `lineitems_clean.parquet` is NOT the source for that — go back to the raw.
5. **No brand / private-label / organic flags exist in the master.** Don't
   expect to be able to enrich with these without an external source.
6. **Date range:** H1 2022 only (Jan 1 – Jun 30). Phase 2's time-decay window
   has to fit in 6 months.

— Data-prep phase, {datetime.utcnow().date().isoformat()}
"""
HANDOFF_MD.write_text(handoff)
print(f"Handoff written → {HANDOFF_MD}")
print(f"  ({len(handoff):,} chars)")
